# R9 · Cross-dataset maps and partial queries from held-out tiles (Fig 4A, B)

* **Skin cells (seed 0):** each programme's cell score and query covariance, each cell's index tile and the tiles Spindle
  returns, for the maps (with `results/pan_cancer/per_dataset/skin_melanoma/seed0_gradient.npz`).
* **Skin gene-set queries (seed 0):** every held-out tile with the accuracy benchmark's gene sets (|S| = 4–32):
  composition difference of the query tile to Spindle's first 10 tiles and to a random 10.
* **Every section (seed 0):** the skin maps' arrays on every section with all programme genes (all but lymph node 5k),
  for the Spearman's ρ table, and the gene-set queries on every section, for the composition table.

The rules are fixed in `experiments/common/skin_maps.py`.

* **Prerequisites:** `R0_index_cache.ipynb`, `R8_pan_cancer.ipynb`.
* **Writes:** `results/skin_maps/`: `skin_cells.npz`, `skin_gene_set_queries.csv`,
  `per_dataset/<key>/seed0_{maps.npz,gene_set_queries.csv}`.
* **Skips** existing outputs (`FORCE = True` redoes all).
* **Runtime / memory:** about 10–20 min on one core; peak about 11 GB.

In [1]:
import gc
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import numpy as np
import pandas as pd

import data
import results_io as rio
import sections as sx
import skin_maps as sm

FORCE = False
TOPIC = "skin_maps"
OUT = rio.topic(TOPIC)
PC = ROOT / "results" / "pan_cancer"
print(data.stamp())
E = pd.read_csv(PC / "exemplar.csv")
Z = dict(np.load(PC / "exemplar.npz"))

{'tag': 'method-v2', 'describe': 'method-v3-dirty', 'commit': 'b35f7d4'}


## Skin, seed 0: cell scores

In [2]:
if FORCE or not (OUT / "skin_cells.npz").exists():
    S = sx.load(sm.KEY, data.EXAMPLE_SEED, parts=("layout", "spindle_partial"))
    path = OUT / "skin_cells.npz"
    np.savez_compressed(path, **sm.skin_cells(S, E, Z))
    rio.register(TOPIC, path, "skin seed 0: cells, cell scores (z-scored genes), tile scores, cell -> index tile, query Q and the tiles Spindle returns, per programme")
    del S
    gc.collect()

## Skin, seed 0: gene-set queries

In [3]:
if FORCE or not (OUT / "skin_gene_set_queries.csv").exists():
    S = sx.load(sm.KEY, data.EXAMPLE_SEED, parts=("layout", "spindle_partial", "D"))
    G = sm.gene_set_queries(S, data.EXAMPLE_SEED)
    rio.write_csv(G, TOPIC, "skin_gene_set_queries.csv",
                  "skin seed 0, every gene-set query: composition difference to Spindle's first 10 and a random 10")
    del S
    gc.collect()
G = pd.read_csv(OUT / "skin_gene_set_queries.csv")
G.groupby("size")[["n_found", "tv_spindle", "tv_random"]].median().round(3)

,n_found,tv_spindle,tv_random
size,,,
4,29.0,0.383,0.570
8,82.0,0.340,0.569
16,187.0,0.359,0.576
32,242.0,0.317,0.570


## Every section, seed 0: the skin maps' arrays and gene-set queries (supplementary tables)

In [4]:
MAP_KEYS = sm.map_keys(data.indexed_keys())
print("maps:", MAP_KEYS)
for key in data.indexed_keys():
    f_maps, f_gs = f"per_dataset/{key}/seed0_maps.npz", f"per_dataset/{key}/seed0_gene_set_queries.csv"
    need_maps = key in MAP_KEYS and (FORCE or not (OUT / f_maps).exists())
    need_gs = FORCE or not (OUT / f_gs).exists()
    if not (need_maps or need_gs):
        print(f"{key}: exists, skipped")
        continue
    S = sx.load(key, data.EXAMPLE_SEED, parts=("layout", "naive_partial", "spindle_partial", "D"))
    if need_maps:
        path = OUT / f_maps
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(path, **sm.section_maps(S, E, Z))
        rio.register(TOPIC, path, "seed 0: cells, cell and tile scores, cell -> index tile, query Q, tiles Spindle returns, "
                                  "exact d_B^S of every index tile, tile centres and sides, per programme")
    if need_gs:
        G = sm.gene_set_queries(S, data.EXAMPLE_SEED)
        rio.write_csv(G, TOPIC, f_gs, "seed 0, every gene-set query: composition difference to Spindle's first 10 "
                                      "and a random 10")
    del S
    gc.collect()
    print(f"{key}: done; peak memory so far {rio.peak_rss_gb():.1f} GB", flush=True)

G = pd.concat([pd.read_csv(OUT / f"per_dataset/{k}/seed0_gene_set_queries.csv") for k in data.indexed_keys()])
G.groupby(["key", "size"])[["n_found", "tv_spindle", "tv_random"]].median().round(3)

maps: ['skin_melanoma', 'kidney_nondiseased', 'breast_cancer', 'lung_cancer', 'pancreatic_cancer', 'lymph_node', 'brain_cancer']
skin_melanoma: exists, skipped
kidney_nondiseased: exists, skipped
breast_cancer: exists, skipped
lung_cancer: exists, skipped
pancreatic_cancer: exists, skipped
lymph_node: exists, skipped
lymph_node_5k: exists, skipped
brain_cancer: exists, skipped


n_found  tv_spindle  tv_random
key                size                                
brain_cancer       4        40.0       0.319      0.358
                   8       129.0       0.311      0.357
                   16      535.5       0.291      0.357
                   32     2708.0       0.243      0.357
breast_cancer      4        35.0       0.378      0.529
                   8       144.5       0.397      0.531
                   16      348.0       0.431      0.528
                   32      674.5       0.356      0.530
kidney_nondiseased 4        57.0       0.332      0.348
                   8       159.5       0.339      0.352
                   16      471.0       0.313      0.352
                   32      683.0       0.340      0.354
lung_cancer        4        41.0       0.386      0.445
                   8       166.5       0.367      0.449
                   16      763.0       0.378      0.451
                   32     1198.5       0.307      0.451
lymph_node         4        33.0       0.366      0.409
                   8       156.0       0.356      0.407
                   16      771.0       0.327      0.410
                   32     1050.0       0.254      0.405
lymph_node_5k      4        34.0       0.369      0.403
                   8       219.5       0.353      0.403
                   16      806.5       0.334      0.403
                   32     1829.0       0.300      0.402
pancreatic_cancer  4        39.0       0.389      0.491
                   8       199.0       0.400      0.491
                   16      511.0       0.381      0.491
                   32      873.0       0.340      0.457
skin_melanoma      4        29.0       0.383      0.570
                   8        82.0       0.340      0.569
                   16      187.0       0.359      0.576
                   32      242.0       0.317      0.570

In [5]:
rio.write_manifest(TOPIC, "R9_skin_maps.ipynb", inputs=["results/_cache", "results/pan_cancer"], notes="Fig 4A, B; supplementary tables.")

wrote results/skin_maps/manifest.json: 17 files; peak memory 0.16 GB


{'topic': 'skin_maps',
 'notebook': 'R9_skin_maps.ipynb',
 'date': '2026-11-05 09:40',
 'tag': 'method-v2',
 'describe': 'method-v3-dirty',
 'commit': 'b35f7d4',
 'inputs': ['results/_cache', 'results/pan_cancer'],
 'peak_rss_gb': 0.16,
 'notes': 'Fig 4A, B; supplementary tables.',
 'files': {'per_dataset/brain_cancer/seed0_gene_set_queries.csv': {'description': "seed 0, every gene-set query: composition difference to Spindle's first 10 and a random 10",
   'rows': 4248,
   'bytes': 348990},
  'per_dataset/brain_cancer/seed0_maps.npz': {'description': 'seed 0: cells, cell and tile scores, cell -> index tile, query Q, tiles Spindle returns, exact d_B^S of every index tile, tile centres and sides, per programme',
   'rows': None,
   'bytes': 9487562},
  'per_dataset/breast_cancer/seed0_gene_set_queries.csv': {'description': "seed 0, every gene-set query: composition difference to Spindle's first 10 and a random 10",
   'rows': 816,
   'bytes': 66632},
  'per_dataset/breast_cancer/seed0_m